### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="sepsis_survival_minimal_clinical_records",
    dataset_year="2020",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C53C8N",
    download_description="""
We download the data from UCI.

wget https://archive.ics.uci.edu/static/public/827/sepsis+survival+minimal+clinical+records.zip && unzip sepsis+survival+minimal+clinical+records.zip && unzip s41598-020-73558-3_sepsis_survival_dataset.zip -d data_files &&  rm sepsis+survival+minimal+clinical+records.zip && rm s41598-020-73558-3_sepsis_survival_dataset.zip
mkdir -p local-data-warehouse/sepsis_survival_minimal_clinical_records && mv data_files local-data-warehouse/sepsis_survival_minimal_clinical_records/
""",
    # References
    academic_reference_bibtex="""@article{chicco2020survival,
  title={Survival prediction of patients with sepsis from age, sex, and septic episode number alone},
  author={Chicco, Davide and Jurman, Giuseppe},
  journal={Scientific reports},
  volume={10},
  number={1},
  pages={17156},
  year={2020},
  publisher={Nature Publishing Group UK London}
}
""",
    academic_reference_bibtex_key="chicco2020survival",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We use only the primary cohort from Norway and ignore the subset and the small data (137 samples) from the South Korea cohort.

- The data has only 3 features. Moreover, any other information (such as timestamps from the collection) are not provided. But this should not limit the applicability of the data, as there should be no temporal leakage.
- We reverse the ordinal encoding of gender and the target variable.
- The data has a lot of naturally occurring duplicates. We keep them in as they make sense they would exist and leave it to the pipeline or methods to handle.
- The alternative subset (the study cohort in the original paper) contains only patients that had a sepsis according to the official Sepsis-3 definition. The paper discusses this in more detail and opts for using/testing all cohorts. We could introduce several datasets from this source but this would bias the benchmark too much towards this specific data distribution. We stick to using all data from the primary cohort.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="sepsis_outcome_after9pt5_days_in_hospital", # we rename the target column during preprocessing
    problem_type="binary_classification",
    objective_metric_name="roc_auc", # paper used various metrics and focused on various thresholds, we stick to ROC AUC for now.
    stratify_on="sepsis_outcome_after9pt5_days_in_hospital",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "data_files" / "s41598-020-73558-3_sepsis_survival_primary_cohort.csv")
print("Loaded data shape:", df.shape)

# Reverse the ordinal encoding
df["gender"] = df["sex_0male_1female"].replace({0: "Male", 1: "Female"}).astype("category")
df["sepsis_outcome_after9pt5_days_in_hospital"] = df["hospital_outcome_1alive_0dead"].replace({0: "Dead", 1: "Alive"}).astype("category")
df = df.drop(columns=["sex_0male_1female", "hospital_outcome_1alive_0dead"])

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()